# Generating input file for bulk mosaic cutouts with Cutana

<br>

<img style="position: absolute; right: 60px; top: 40px; height: 160px; width: 160px" src="https://datalabs.esa.int/datalab-icon/c11e47af-7ce9-4281-b3b5-475a180a256f" /> 


<div style="width: 70%;">

Copyright (c) European Space Agency, 2026.


</div>

**Notebook author(s):** Antonio La Marca

**Last modified:** 06.10.2026

**Notebook summary**

This example notebook demonstrates a simple example of how to generate the input file for using the Cutana software installed in this datalab to make bulk cutouts of Euclid mosaics. Cutana is a high-performance Python pipeline for creating astronomical image cutouts from large FITS tile collections, designed for efficient processing of ESA Datalabs and other astronomical datasets. First in this notebook we demonstrate how to generate your own catalogue input file in the format Cutana expects from an existing user source catalogue file. In the second part, we show how to create the Cutana input file from an example MER query. Here is an overview of what is covered:

1. Generating your own Cutana format input file from an existing user source catalogue file
2. Generating your own Cutana format input file from a MER query
3. (Optional) Adding variable cutout size to the input file

**Useful links:**

* [Cutana documentation](~/Notebooks/Cutana/README.md)

**Running the notebook:** 

* This notebook has **read-only permissions**. You can still run and edit cells but if you would like to save your changes then you have to save the notebook to a different location - your workspace for example. 
* The file generation part of this notebook is intended to be run using the **EUCLID-TOOLS kernel** that has all the necessary packages included. 
***

# Import 

In [ ]:
# to make sure the cutout_utils.py file is updated here after every change
%load_ext autoreload
%autoreload 2

In [ ]:
# importing necessary packages
from astroquery.esa.euclid.core import EuclidClass
import glob
import os
import getpass
import pandas as pd
from astropy.coordinates import SkyCoord

# Functions

Next we have some functions to create the input file for running Cutana. You will need to run this cell for creating the input file. The following functions are designed to generate the specific input file required by Cutana for making Euclid cutouts in IDR and PDR environments. However, the user can customise the functions to match your specific needs. 

The user can also copy and paste the functions in a separate python script and import such a script. 

In [ ]:
# choosing environment
Euclid = None


def set_env(envname, user="", password=""):
    global Euclid
    Euclid = EuclidClass(environment=envname)
    Euclid.login(user=user, password=password)


# Define functions


def chooseBestFile(df_res_dup):
    """Choosing best file for sources with multiple by checking where the source is closest to image centre"""

    min_dist_per_plotcode = df_res_dup.groupby(
        ["right_ascension", "filter_name"], as_index=False, sort=False
    )["dist"].idxmin()
    df_final = df_res_dup.loc[min_dist_per_plotcode["dist"]]
    return df_final


def addFilters(query, instrument_name, nisp_filters=[]):
    """In case nisp_filters is specified modify query to get files for only those filters"""
    filter_part = ""
    query_parts = query.split("ON ")
    for one_filter in nisp_filters:
        if one_filter != nisp_filters[-1]:
            filter_part += "(mosaics.filter_name='" + one_filter + "') OR "
        else:
            filter_part += "(mosaics.filter_name='" + one_filter + "')"
    query = query_parts[0] + "ON (" + filter_part + ") AND" + query_parts[1]
    return query


def addInstrument(query, instrument_name):
    """Add instrument filter to query. instrument_name can be a string ("VIS", "NISP", "") or a list of instruments"""

    # Handle both string and list inputs
    if isinstance(instrument_name, str):
        instruments = [instrument_name] if instrument_name != "" else []
    else:
        instruments = list(instrument_name)

    query_parts = query.split("ON (")

    if len(instruments) == 0:
        # No instrument specified - include both VIS and NISP
        query = (
            query_parts[0]
            + "ON ((mosaics.instrument_name='VIS' OR mosaics.instrument_name='NISP') AND "
            + query_parts[1]
        )
    elif len(instruments) == 1:
        # Single instrument
        if instruments[0] not in ["VIS", "NISP"]:
            raise ValueError(
                f"Incorrect value for parameter instrument_name: {instruments[0]}. Valid values are 'VIS' or 'NISP'"
            )
        query = (
            query_parts[0]
            + "ON (mosaics.instrument_name='"
            + instruments[0]
            + "' AND "
            + query_parts[1]
        )
    else:
        # Multiple instruments
        for instr in instruments:
            if instr not in ["VIS", "NISP"]:
                raise ValueError(
                    f"Incorrect value in instrument list: {instr}. Valid values are 'VIS' or 'NISP'"
                )
        instrument_part = " OR ".join(
            [f"mosaics.instrument_name='{instr}'" for instr in instruments]
        )
        query = query_parts[0] + "ON ((" + instrument_part + ") AND " + query_parts[1]

    return query


def checkInput(cutout_size, instrument_name, nisp_filters):
    """Checking user input value correctness for cutout_size, instrument name and nisp filters.
    instrument_name can be a string or a list of strings"""

    valid_cutout_sizes = ["pixel", "arcsec"]
    valid_instruments = ["VIS", "NISP"]
    valid_filters = ["NIR_H", "NIR_J", "NIR_Y", ""]

    if cutout_size not in valid_cutout_sizes:
        raise ValueError(f"Invalid cutout size. Choose from: {valid_cutout_sizes}")

    # Handle both string and list inputs for instrument_name
    if isinstance(instrument_name, str):
        instruments = [instrument_name] if instrument_name != "" else []
    else:
        instruments = list(instrument_name)

    for instr in instruments:
        if instr not in valid_instruments:
            raise ValueError(
                f"Invalid instrument name. Choose from: {valid_instruments}"
            )

    for filter_name in nisp_filters:
        if filter_name not in valid_filters:
            raise ValueError(f"Invalid filter name. Choose from: {valid_filters}")

    # Check for invalid combination: NISP filters without NISP instrument
    if instruments and nisp_filters != []:
        if "NISP" not in instruments and len(instruments) > 0:
            raise ValueError(
                f"Invalid combination of input parameters. NISP filters specified but NISP instrument not included"
            )

    return


def sortFilePathsByFilterPriority(
    file_paths_list, filter_priority=["NIR_H", "NIR_J", "VIS", "NIR_Y"]
):
    """Sort a list of file paths according to filter priority order.

    Parameters:
    -----------
    file_paths_list : list
        List of file paths to sort
    filter_priority : list
        Order in which filters should be sorted (default: ["NIR_H", "NIR_J", "VIS", "NIR_Y"])

    Returns:
    --------
    list
        Sorted list of file paths
    """

    def get_sort_key(filepath):
        """Extract filter name from filepath and return its priority index"""
        for priority_idx, filter_name in enumerate(filter_priority):
            if filter_name in filepath:
                return priority_idx
        # If no filter found in priority list, put at the end
        return len(filter_priority)

    return sorted(file_paths_list, key=get_sort_key)


def generateCutanaInputFile(
    source_query,
    instrument_name="",
    nisp_filters=[],
    cutout_size="pixel",
    cutout_size_value=50,
    output_filepath="/media/user/example_notebook_outputs/cutana_input.csv",
    drop_noncutana_cols=True,
    env="IDR",
):
    """Match sources with mosaic files by checking if source coordinates are in mosaic file fov or
    if file tile_index matches the first 9 digits of source's segmentation_map_id.

    Parameters:
    -----------
    source_query : str
        Query string or user table name for sources
    instrument_name : str or list, optional
        Single instrument ("VIS" or "NISP"), multiple instruments (list), or "" for both (default: "")
        Examples: "VIS", "NISP", ["VIS", "NISP"], ""
    nisp_filters : list, optional
        List of NISP filters (e.g., ["NIR_Y", "NIR_H"]) - only valid with NISP instrument
    cutout_size : str, optional
        Unit for cutout size ("pixel" or "arcsec", default: "pixel")
    cutout_size_value : int, optional
        Size of cutout (default: 50)
    output_filepath : str, optional
        Path to save the output CSV file
    drop_noncutana_cols : bool, optional
        Whether to drop non-Cutana columns (default: True)
    env : str, optional
        Archive environment ("IDR" or "PDR", default: "IDR")

    Returns:
    --------
    pandas.DataFrame
        DataFrame with Cutana-compatible format
    """

    # check input correctness
    checkInput(cutout_size, instrument_name, nisp_filters)

    shortcut = False

    if env == "IDR":
        dr = "dr1"
    elif env == "PDR":
        dr = "q1"

    query = f"""SELECT 'TILE_'||CAST(mosaics.tile_index AS CHAR(25))||'_'||CAST(sources.right_ascension AS CHAR(25)) as SourceID, sources.right_ascension, sources.declination, mosaics.datalabs_path||'/'||mosaics.file_name as fits_file_paths, mosaics.filter_name, mosaics.instrument_name, DISTANCE(mosaics.ra, mosaics.dec, sources.right_ascension, sources.declination) AS dist
                FROM {source_query} as sources
                JOIN {dr}.mosaic_product AS mosaics
                ON (mosaics.fov IS NOT NULL AND CONTAINS(CIRCLE(sources.right_ascension, sources.declination, {0.5/60}), mosaics.fov)=1)
                WHERE mosaics.datalabs_path IS NOT NULL
                ORDER BY sources.right_ascension ASC"""
    print(query)

    # check if column names contain segmenation_map_id and object_id for shortcut
    if "user" in source_query:
        all_colnames = [
            col.name
            for col in Euclid.load_table(table=source_query, verbose=False).columns
        ]
    else:
        all_colnames = source_query

    if "segmentation_map_id" in all_colnames:
        # print('SHORTCUT')
        shortcut = True
        query = f"""SELECT 'TILE_'||SUBSTRING(CAST(segmentation_map_id AS CHAR(25)), 1, 9)||'_'||CAST(sources.right_ascension AS CHAR(25)) as SourceID, sources.right_ascension, sources.declination, mosaics.datalabs_path||'/'||mosaics.file_name as fits_file_paths, mosaics.filter_name, mosaics.instrument_name
            FROM {source_query} as sources
            JOIN mosaic_product AS mosaics
            ON (CAST(mosaics.tile_index AS CHAR(25)) = SUBSTRING(CAST(segmentation_map_id AS CHAR(25)), 1, 9))
            WHERE mosaics.datalabs_path IS NOT NULL
            ORDER BY sources.right_ascension ASC"""

    # modify query to include instrument selection
    query = addInstrument(query, instrument_name)

    # modify query to include only the provided nisp filters
    if len(nisp_filters) > 0 and len(nisp_filters) < 3:
        query = addFilters(query, instrument_name, nisp_filters)

    # make query to get file info
    job = Euclid.launch_job_async(query, verbose=False)
    if job == None:
        raise AttributeError(
            "Query for files failed. Please make sure your source query or user table contains columns named 'right_ascension' and 'declination' or 'ra' and 'dec'. When using a source query it should have brackets around it."
        )
    df_res = job.get_results().to_pandas()

    # choose best mosaic
    if shortcut == False:
        df_res = chooseBestFile(df_res)

    df_res = df_res.replace({"SourceID": {"\.": ""}}, regex=True)

    # Group by coordinates, filter, and instrument to preserve separation of instrument data
    group_cols = list(df_res)[: list(df_res).index("fits_file_paths")]
    df_res = (
        df_res.groupby(group_cols, sort=False)["fits_file_paths"]
        .apply(lambda grp: list(set(grp)))
        .reset_index()
    )
    print("Found", len(df_res), "query results")

    # Sort the file paths within each row according to filter priority
    df_res["fits_file_paths"] = df_res["fits_file_paths"].apply(
        lambda x: sortFilePathsByFilterPriority(
            x, filter_priority=["NIR_H", "NIR_J", "VIS", "NIR_Y"]
        )
    )

    # add constant diameter
    df_res.insert(loc=3, column="diameter_" + cutout_size, value=cutout_size_value)

    # convert list of file_paths to str
    df_res = df_res.astype({"fits_file_paths": str})

    # add user columns back if requested
    if drop_noncutana_cols == False:
        query_user_table = f"""SELECT * FROM {source_query} as sources"""
        job = Euclid.launch_job_async(query_user_table, verbose=False)
        df_user_table = job.get_results().to_pandas()
        df_res = pd.merge(
            df_res, df_user_table, on=["right_ascension", "declination"], how="outer"
        )

    # rename cols to fit cutana
    df_res = df_res.rename(columns={"right_ascension": "RA", "declination": "Dec"})

    # Drop instrument_name column as it's not needed in final output
    if "instrument_name" in df_res.columns:
        df_res = df_res.drop(columns=["instrument_name"])

    # save to csv file for cutana
    df_res.to_csv(output_filepath, index=False)
    print("Creating Cutana compatible input file at", output_filepath)

    return df_res


def get_query_results(query, n_rows=3):
    """"""
    "Display ADQL query and first results"
    ""
    print(f"ADQL query: ")
    print(query)
    print()
    job = Euclid.launch_job_async(query)
    res = job.get_results()
    print(f"* Output table contains {len(res):,} rows")
    print(f"* Displaying first {n_rows} rows: ")
    display(res[0:n_rows])
    return res

### Choosing the archive environment

Note that **this notebook can work with multiple archive environments: IDR, PDR and REG** (also OTF once there is data). In the cell below you can **choose the archive environment** you want to work with (default is IDR).

Note: this notebook requires to login.

In [ ]:
# choosing the archive envionment
env = "IDR"  # can be changed to 'IDR', 'PDR' or 'REG' too

# setting the environment name for the notebook
Euclid = EuclidClass(environment=env)

# Login ============================
user = input(
    "Enter your user name: "
)  # Alternatively, you may set your user in the line below.
# user     = ''
password = getpass.getpass(prompt="Enter your password: ")
Euclid.login(user=user, password=password)


# setting the environment name in the backend too
set_env(env, user=user, password=password)

# 1. Generate a Cutana input file from an existing source file

### Getting sources from an existing user file

You may want to use a source table that exists in your workspace in ESA Datalabs. The source table in your file needs to have columns called either "right_ascension" and "declination" or "ra" and "dec" for the source coordinates. It may have any number of other columns as well. Optional, but reccomended, is the inclusion of an identifier (`id`) column. 

This example requires some user interaction from your side. In order to use a file you already have you first need to upload it to ESA Datalabs and then in the cells below provide a path to where you uploaded your file. For this example, we provide a list of sources in the file `user_source_input_edfn.csv`

The upload function allows formats like: fits, csv, ecsv, VOTable, ASCII and when naming your table please avoid using upper case letters.

In [ ]:
# Input file with user sources
input_filepath = "user_source_input_edfn.csv"  # Insert the user input file path

# upload file as user table directly
Euclid.upload_table(
    upload_resource=input_filepath,
    table_name="my_sources_from_file_tutorial",  # Insert table name
    format="csv",
)

**Note**: if the table name already exists, you will need to delete the previous table and re-upload the source file. Alternatively, you can set a different name. To delete the already existing file, use the following code:

`# Delete table if it already exists
Euclid.delete_user_table(table_name=user_table_name)`

where `user_table_name` is in the form "user_johnsmith.mysources"

### Generate the Cutana input file

Here we generate the Cutana input file. The user must specify:

* The instrument name ("VIS" or "NISP")
* The NISP filters in case of "NISP" instrumnet ("NISP_Y", "NISP_J", "NIR_Y"). Multiple filters are possible. 
* Cutouts size

In [ ]:
# Specify the user table name uploaded
user_table = (
    "user_johnsmith.my_sources_from_file_tutorial"  # (e.g. "user_johnsmith.mysources")
)

# Generate cutana input file
df_cutana_file = generateCutanaInputFile(
    user_table,
    cutout_size_value=200,
    instrument_name = ["VIS", "NISP"],  # "VIS" or "NISP" or ["VIS", "NISP"]
    nisp_filters = ["NIR_Y", "NIR_J", "NIR_H"], # Or leave it blank if only VIS
    env = env,
)  # Indicate list of filters
df_cutana_file

### Save the output file

In [ ]:
# Output file path
output_filepath = "user_source_cutana_input.csv"  # Insert output filename

# Save the output table to a `csv` file
df_cutana_file.to_csv(
    path_or_buf=output_filepath,
    index=False,
)

With the saved table, you can run Cutana to generate bulk mosaic cutouts using the example notebook: [Bulk Mosaic Cutouts](~/Notebooks/Cutana/Bulk_mosaic_cutouts.ipynb).

### Delete the user table from the archive

This is not a mandatory step, but to keep the personal archive space clean we suggest to delete the table just created. However, if the user plan to re-use the uploaded source table, this cell can be skipped. 

In [ ]:
# Delete existing table
Euclid.delete_user_table(table_name="user_johnsmith.my_sources_from_file_tutorial")

# 2. Generate a Cutana input file from a MER source query

In this second part, we showcase how to generate a Cutana input file from a Euclid MER source query. In this case, we query some Euclid data of which we would like to make cutouts and produce directly the file to use as input in Cutana. 

### Setting up a MER source query

For this example, we would like to query some Euclid sources that are likely galaxies, in the redshift range 0.5<z<1.0, in the euclid deep field north (EDF-N) area. In particular, we select only the 100 brightest objects. 

First we run the desired query to check that the output is what we expect; then we will generate the Cutana file. 

In [ ]:
Euclid.login(user=user, password=password)

# setting the environment name in the backend too
set_env(env, user=user, password=password)

In [ ]:
inp_rad = 2  # Cone search radius in degrees - it can be updated as needed

edfn_c = SkyCoord(
    "17h58m55.9s", "66d01m03.7s", frame="icrs"
)  # EDF-N central coordinates
inp_coords = [edfn_c.ra.deg, edfn_c.dec.deg]  # RA & Dec - it can be updated as needed

if env == "IDR":
    table_1 = "mer_catalogue_deep_survey"
    table_2 = "phz_classification_deep_survey"
    table_3 = "phz_physical_parameters_deep_survey"
elif env == "PDR":
    table_1 = "mer_catalogue"
    table_2 = "phz_classification"
    table_3 = "phz_physical_parameters"

# print(df_env['Title'][query_i])
# print()
source_query = f"""
    (SELECT TOP 100
    mer.object_id,
    mer.right_ascension,
    mer.declination,
    mer.flux_detection_total,
    phz_phys.phz_pp_median_redshift
    FROM catalogue.{table_1} AS mer
    JOIN catalogue.{table_2} AS phz_class USING (object_id)
    JOIN catalogue.{table_3} AS phz_phys USING (object_id)
    WHERE DISTANCE({inp_coords[0]}, {inp_coords[1]}, mer.right_ascension, mer.declination) < {inp_rad}
    AND phz_class.phz_gal_prob > 0.9
    AND phz_phys.phz_pp_median_redshift > 0.5
    AND phz_phys.phz_pp_median_redshift < 1.0
    ORDER BY mer.flux_detection_total DESC)
"""

res = get_query_results(
    source_query, n_rows=3
)  # Display the results of the query. Change the number of rows as needed

### Generating the Cutana input file

After checking that the query output is as required, we can now generate the Cutana input file. In thise case, we use again the `generateCutanaInputFile` function, but passing the source query rather than a user table as done in the first example. 

**Note**: your query must include brackets '()' to run correctly.

In [ ]:
# Generate cutana input file
df_cutana_file = generateCutanaInputFile(
    source_query,
    cutout_size_value=200,
    instrument_name="VIS",  # VIS or NISP or ["VIS", "NISP"]
    nisp_filters = ["NIR_Y", "NIR_J", "NIR_H"], # Or leave it blank if only VIS
    env=env,
)  # Indicate list of filters
df_cutana_file

### Save the output file

In [ ]:
# Output file path
output_filepath = "user_query_cutana_input.csv"  # Insert output filename

# Save the output table to a `csv` file
df_cutana_file.to_csv(
    path_or_buf=output_filepath,
    index=False,
)

With the saved table, you can run Cutana to generate bulk mosaic cutouts using the example notebook: [Bulk Mosaic Cutouts](~/Notebooks/Cutana/Bulk_mosaic_cutouts.ipynb).
